In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()
MAIN_ASSET = os.getenv("MAIN_ASSET", "EURUSD")

class Backtester:
    """Run backtest using the Multi-Class model and trading environment."""

    def __init__(self, model, environment, threshold=0.35, risk_percentage=0.2):
        self.model = model
        self.environment = environment
        
        # 3-class baseline is 0.33. 0.35 threshold ensures slight conviction.
        self.threshold = threshold 
        self.risk_percentage = risk_percentage 
        
        # FIX: Align Backtester with the Preprocessor's ATR Logic
        self.tp_mult = 1.5
        self.sl_mult = 0.75

        self.cooldown_periods = 2 # Reduced cooldown for faster 6h strategy
        self._cooldown_counter = 0

    def run(self, data_path, asset_price_col='Close'):
        print(f"Starting Multi-Class Dynamic ATR Backtest on {data_path}...")
        df = pd.read_csv(data_path, index_col=0, parse_dates=True)
        ticker = MAIN_ASSET

        if 'target' in df.columns:
            features_df = df.drop(columns=['target'])
        else:
            features_df = df

        results = []
        self.environment.reset()
        self._cooldown_counter = 0
        
        seq_len = self.model.input_chunk_length
        current_sl_threshold = 0.0

        for i in range(seq_len, len(df)):
            current_row = df.iloc[i]
            current_timestamp = df.index[i]
            price = current_row[asset_price_col]
            
            # Extract ATR for dynamic logic (fallback to 0.5% move if missing)
            atr = current_row.get('ATR', price * 0.005) 
            
            self.environment.update_portfolio_value({ticker: price})
            port_value = self.environment.portfolio_value
            current_pos = self.environment.positions.get(ticker, 0.0)
            entry_price = self.environment.entry_prices.get(ticker, 0.0)

            if self._cooldown_counter > 0:
                self._cooldown_counter -= 1

            # Reset dynamic Stop Loss if flat
            if current_pos == 0:
                current_sl_threshold = -((self.sl_mult * atr) / price)

            # 1. Evaluate Dynamic ATR Risk Management
            closed_due_to_risk = False
            if current_pos != 0 and entry_price > 0:
                
                # Convert the ATR dollar move into a percentage for the trailing math
                tp_pct = (self.tp_mult * atr) / entry_price
                base_sl_pct = -((self.sl_mult * atr) / entry_price)
                
                unrealized_return = (price - entry_price) / entry_price if current_pos > 0 else (entry_price - price) / entry_price
                
                # Rule 2: Break-even Plus
                if unrealized_return >= 0.70 * tp_pct:
                    locked_in_profit = 0.50 * tp_pct
                    if locked_in_profit > current_sl_threshold:
                        current_sl_threshold = locked_in_profit
                
                if unrealized_return <= current_sl_threshold or unrealized_return >= tp_pct:
                    self.environment.execute_trade(ticker, -current_pos, price, current_timestamp)
                    self._cooldown_counter = self.cooldown_periods
                    closed_due_to_risk = True
                    current_pos = 0.0 
                    current_sl_threshold = base_sl_pct 

            # 2. Make Predictions & Issue Orders
            if not closed_due_to_risk and self._cooldown_counter == 0:
                sequence = features_df.iloc[i - seq_len : i].values
                
                probs = self.model.predict(sequence)
                p_flat, p_up, p_down = probs[0], probs[1], probs[2]

                target_pos = current_pos

                # Multi-class Execution Logic
                if p_up > self.threshold and p_up > p_down:
                    target_pos = 1.0  # Confident Up
                elif p_down > self.threshold and p_down > p_up:
                    target_pos = -1.0 # Confident Down
                elif p_flat > max(p_up, p_down):
                    target_pos = 0.0  # Volatile Chop/Sideways -> Exit
                
                # Execute difference
                if target_pos != current_pos:
                    trade_size = target_pos - current_pos
                    self.environment.execute_trade(ticker, trade_size, price, current_timestamp)
                    current_pos = target_pos

            results.append({
                'timestamp': current_timestamp,
                'price': price,
                'position': current_pos,
                'portfolio_value': port_value,
                'prob_flat': probs[0],
                'prob_up': probs[1],
                'prob_down': probs[2]
            })

        return pd.DataFrame(results)

In [ ]:
import os
import pandas as pd
from pathlib import Path
from zipfile import ZipFile
from dotenv import load_dotenv
from datetime import datetime

try:
    from histdata import download_hist_data
except ImportError:
    print("[ERROR] Please install the histdata package: pip install histdata")
    exit(1)

# Load environment variables
load_dotenv()

# Configuration from .env
TICKERS_ENV = os.getenv("HISTDATA_TICKERS", "EURUSD,GBPUSD,USDJPY")
TICKERS = [t.strip() for t in TICKERS_ENV.split(",")]
START_YEAR = int(os.getenv("HISTDATA_START_YEAR", 2008))
END_YEAR = int(os.getenv("HISTDATA_END_YEAR", 2026))

# Ensure data directory exists
DATA_PATH = Path("trading_bot/data/raw")
DATA_PATH.mkdir(parents=True, exist_ok=True)

def download_histdata(tickers, start_year, end_year, path):
    """
    Downloads 1-minute FX data from HistData.com and saves it directly.
    Resampling is deferred to the preprocessing stage.
    """
    print("Downloading Raw 1-Minute Data from HistData.com")
    print(f"Time Range: {start_year} to {end_year}\n")

    current_date = datetime.now()
    current_year_actual = current_date.year
    current_month_actual = current_date.month

    for ticker in tickers:
        print(f"Fetching {ticker}...")
        df_list = []
        
        # HistData expects lowercase, no-symbol tickers (e.g., 'eurusd')
        clean_ticker = ticker.replace("/", "").replace("-", "").replace("_", "").lower()
        
        for year in range(start_year, end_year + 1):
            if year > current_year_actual:
                print(f"  Skipping {year} (Future Year)")
                continue
                
            # If it's the current ongoing year, HistData requires month-by-month downloads
            months_to_fetch = [None] 
            if year == current_year_actual:
                months_to_fetch = list(range(1, current_month_actual + 1))

            for month in months_to_fetch:
                try:
                    if month is None:
                        print(f"  Downloading {year}...")
                    else:
                        print(f"  Downloading {year}-{month:02d}...")
                    
                    # Download GENERIC_ASCII 1-Minute data
                    zip_path = download_hist_data(
                        year=str(year), 
                        month=str(month) if month else None,
                        pair=clean_ticker, 
                        time_frame='M1',
                        platform='ASCII',
                        output_directory=str(path)
                    )
                    
                    if zip_path:
                        with ZipFile(zip_path, 'r') as z:
                            csv_name = z.namelist()[0]
                            with z.open(csv_name) as f:
                                # GENERIC ASCII format: YYYYMMDD HHMMSS;Open;High;Low;Close;Volume
                                df_year = pd.read_csv(
                                    f, 
                                    sep=';', 
                                    names=['datetime', 'Open', 'High', 'Low', 'Close', 'Volume'], 
                                    header=None
                                )
                                
                                # Parse dates and set index
                                df_year['datetime'] = pd.to_datetime(df_year['datetime'], format='%Y%m%d %H%M%S')
                                df_year.set_index('datetime', inplace=True)
                                df_list.append(df_year)
                                
                        # Clean up the zip file from the raw folder after reading into memory
                        os.remove(zip_path)
                        
                except Exception as e:
                    msg = f"Failed to fetch {ticker} for {year}" + (f"-{month:02d}" if month else "")
                    print(f"[WARNING] {msg}: {e}")
                    if 'zip_path' in locals() and zip_path and os.path.exists(zip_path):
                        os.remove(zip_path)

        if df_list:
            print("  Combining 1-Minute data...")
            # Combine all years into one massive DataFrame
            full_df = pd.concat(df_list)
            full_df.sort_index(inplace=True)
            full_df.index.name = 'Date'
            
            # Save the raw 1-minute data directly to the raw directory
            out_file = path / f"{ticker.upper()}.csv"
            full_df.to_csv(out_file)
            print(f"[OK] Saved {len(full_df)} raw 1-minute candles to {out_file}\n")
        else:
            print(f"[WARNING] No data could be processed for {ticker}.\n")

if __name__ == "__main__":
    download_histdata(TICKERS, START_YEAR, END_YEAR, DATA_PATH)

In [ ]:
import numpy as np
import pandas as pd

class TradingEnvironment:
    """Custom environment for simulating trading with a portfolio."""

    def __init__(self, initial_capital=10000):
        self.initial_capital = initial_capital
        self.capital = initial_capital
        self.positions = {}  # {ticker: quantity}
        self.entry_prices = {} # {ticker: avg_entry_price}
        self.portfolio_value = initial_capital
        self.trade_history = []
        self.portfolio_history = [initial_capital]

    def get_portfolio_value(self, prices):
        """Calculate current portfolio value given current prices."""
        cash = self.capital
        for ticker, qty in self.positions.items():
            if ticker in prices:
                cash += qty * prices[ticker]
        return cash

    def get_fee_rate(self, ticker):
        """FIX: Dynamic fee structure: lower for Forex, standard for Crypto."""
        # Simple heuristic to classify the asset class by ticker name
        if "USD" in ticker and "BTC" not in ticker and "ETH" not in ticker:
            return 0.0001  # 0.01% representing tight spreads for Forex/Fiat (e.g. EURUSD)
        return 0.001       # 0.1% representing standard exchange fees for Crypto/Equities

    def execute_trade(self, ticker, quantity, price, timestamp):
        """Execute a trade (buy/long if positive quantity, sell/short if negative)."""
        if quantity == 0:
            return False

        cost = abs(quantity * price)
        fee = cost * self.get_fee_rate(ticker)

        # Prevent trading if bankrupt
        if self.portfolio_value <= 0:
            return False

        # FIX: Execution logic properly manages bidirectional capital flow
        if quantity > 0:  # Buy / Go Long or Cover Short
            self.capital -= (cost + fee)
        elif quantity < 0:  # Sell / Go Short or Close Long
            self.capital += (cost - fee)

        # Update position
        old_qty = self.positions.get(ticker, 0.0)
        new_qty = old_qty + quantity
        self.positions[ticker] = new_qty

        # FIX: Update entry price for accurate PnL & Stop-loss tracking
        if new_qty == 0:
            self.entry_prices[ticker] = 0.0
        elif (old_qty >= 0 and quantity > 0) or (old_qty <= 0 and quantity < 0):
            # Adding to a position in the same direction: Calculate Volume Weighted Average Price (VWAP)
            old_value = abs(old_qty) * self.entry_prices.get(ticker, price)
            new_value = abs(quantity) * price
            self.entry_prices[ticker] = (old_value + new_value) / abs(new_qty)
        elif (old_qty > 0 > new_qty) or (old_qty < 0 < new_qty):
            # Flipped position entirely (e.g., long reversed to short)
            self.entry_prices[ticker] = price
        # Else: Partially closing a position; the average entry price remains the same.

        self.trade_history.append({
            'timestamp': timestamp,
            'ticker': ticker,
            'quantity': quantity,
            'price': price,
            'fee': fee,
        })
        return True

    def update_portfolio_value(self, prices):
        """Update portfolio value at the end of each time step."""
        self.portfolio_value = self.get_portfolio_value(prices)
        self.portfolio_history.append(self.portfolio_value)

    def close_all_positions(self, prices, timestamp):
        """Close all open positions at market prices."""
        for ticker in list(self.positions.keys()):
            qty = self.positions[ticker]
            if qty != 0:
                self.execute_trade(ticker, -qty, prices[ticker], timestamp)

    def reset(self):
        """Reset the environment."""
        self.capital = self.initial_capital
        self.positions = {}
        self.entry_prices = {}
        self.portfolio_value = self.initial_capital
        self.trade_history = []
        self.portfolio_history = [self.initial_capital]

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
import joblib
import os
from pathlib import Path

class TemporalFusionTransformer(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super(TemporalFusionTransformer, self).__init__()
        
        # LSTM Encoder
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Temporal Attention Layer
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=4,
            dropout=dropout,
            batch_first=True
        )
        
        # 3-Class Classifier (0: Flat, 1: Up, 2: Down)
        # Note: No Sigmoid/Softmax here; nn.CrossEntropyLoss expects raw logits
        self.fc_out = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 3) 
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        final_timestep = attn_out[:, -1, :]
        out = self.fc_out(final_timestep)
        return out

class PricePredictor:
    def __init__(self, input_chunk_length=30, hidden_size=64, num_layers=2):
        self.input_chunk_length = input_chunk_length
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.scaler = StandardScaler()
        self.model = None
        self.history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
        
        # For Confusion Matrix
        self.last_val_targets = []
        self.last_val_preds = []

    def prepare_data(self, csv_path, is_training=True):
        df = pd.read_csv(csv_path, index_col=0, parse_dates=True)
        
        # Separate features and target
        target = df['target'].values
        # Ensure features is strictly a numpy array to avoid sklearn warnings
        features = df.drop(columns=['target']).values

        if is_training:
            features_scaled = self.scaler.fit_transform(features)
        else:
            features_scaled = self.scaler.transform(features)

        X, y = [], []
        for i in range(len(features_scaled) - self.input_chunk_length):
            X.append(features_scaled[i:i + self.input_chunk_length])
            y.append(target[i + self.input_chunk_length])

        return torch.FloatTensor(np.array(X)), torch.LongTensor(np.array(y))

    def train(self, train_csv, epochs=150, batch_size=64):
        X, y = self.prepare_data(train_csv, is_training=True)
        
        # 80/20 Internal Train/Val split
        split = int(0.8 * len(X))
        X_train, y_train = X[:split].to(self.device), y[:split].to(self.device)
        X_val, y_val = X[split:].to(self.device), y[split:].to(self.device)

        self.model = TemporalFusionTransformer(
            input_size=X.shape[2],
            hidden_size=self.hidden_size,
            num_layers=self.num_layers
        ).to(self.device)

        # CrossEntropyLoss automatically handles multi-class outputs
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.model.parameters(), lr=0.001)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

        best_val_loss = float('inf')
        patience_counter = 0
        early_stopping_patience = 15

        print(f"Training Multi-Class TFT (Features: {X.shape[2]}, Target Classes: 3)...")
        for epoch in range(epochs):
            self.model.train()
            train_loss = 0
            
            # Shuffle Training Data
            indices = torch.randperm(X_train.size(0))
            X_train = X_train[indices]
            y_train = y_train[indices]

            for i in range(0, len(X_train), batch_size):
                batch_X = X_train[i:i + batch_size]
                batch_y = y_train[i:i + batch_size]

                optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            train_loss /= (len(X_train) / batch_size)

            # Validation
            self.model.eval()
            val_loss = 0
            correct = 0
            all_preds, all_targets = [], []
            
            with torch.no_grad():
                for i in range(0, len(X_val), batch_size):
                    batch_X = X_val[i:i + batch_size]
                    batch_y = y_val[i:i + batch_size]
                    
                    outputs = self.model(batch_X)
                    loss = criterion(outputs, batch_y)
                    val_loss += loss.item()
                    
                    # Calculate Multiclass Accuracy
                    _, predicted = torch.max(outputs, 1)
                    correct += (predicted == batch_y).sum().item()
                    
                    all_preds.extend(predicted.cpu().numpy())
                    all_targets.extend(batch_y.cpu().numpy())

            val_loss /= (len(X_val) / batch_size)
            val_acc = correct / len(X_val)
            scheduler.step(val_loss)

            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_acc)

            # Save state for confusion matrix
            self.last_val_targets = all_targets
            self.last_val_preds = all_preds

            if epoch % 5 == 0 or epoch == 0:
                lr = optimizer.param_groups[0]['lr']
                print(f"  Epoch {epoch+1:03d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | LR: {lr:.6f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_weights = self.model.state_dict()
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= early_stopping_patience:
                print(f"\nEarly stopping triggered at epoch {epoch+1}. Restoring best weights.")
                self.model.load_state_dict(best_weights)
                break

        return train_loss, val_acc

    def predict(self, sequence):
        self.model.eval()
        with torch.no_grad():
            # FIX: Strip feature names if it's a Pandas DataFrame to suppress sklearn warnings
            if isinstance(sequence, (pd.DataFrame, pd.Series)):
                sequence = sequence.values
            elif not isinstance(sequence, np.ndarray):
                sequence = np.asarray(sequence)
                
            sequence_scaled = self.scaler.transform(sequence)
            seq_tensor = torch.FloatTensor(sequence_scaled).unsqueeze(0).to(self.device)
            outputs = self.model(seq_tensor)
            # Apply softmax to get pure probabilities for [P_Flat, P_Up, P_Down]
            probs = torch.softmax(outputs, dim=1)
            return probs.cpu().numpy()[0]

    def save(self, filepath):
        path_obj = Path(filepath)
        path_obj.parent.mkdir(parents=True, exist_ok=True)
        torch.save(self.model.state_dict(), str(path_obj))
        scaler_path = path_obj.with_suffix('.scaler.pkl')
        joblib.dump(self.scaler, str(scaler_path))

    def plot_confusion_matrix(self, save_path=None):
        if not self.last_val_targets:
            print("No validation data available for Confusion Matrix.")
            return

        cm = confusion_matrix(self.last_val_targets, self.last_val_preds, labels=[0, 1, 2])
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=['Flat (0)', 'Up (+1)', 'Down (-1)'],
                    yticklabels=['Flat (0)', 'Up (+1)', 'Down (-1)'])
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        plt.title('Triple Barrier Confusion Matrix')
        plt.tight_layout()
        
        if save_path:
            path_obj = Path(save_path).resolve()
            path_obj.parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(str(path_obj), dpi=100)
            print(f"Confusion Matrix saved to {path_obj}")
        # plt.show()
        plt.close()

    def plot_training_history(self, save_path=None):
        plt.figure(figsize=(10, 6))
        plt.plot(self.history['train_loss'], label='Train Loss')
        plt.plot(self.history['val_loss'], label='Validation Loss')
        plt.title('Multi-Class Model Training History')
        plt.xlabel('Epoch')
        plt.ylabel('Cross-Entropy Loss')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        if save_path:
            path_obj = Path(save_path).resolve()
            path_obj.parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(str(path_obj), dpi=100)
        # plt.show()
        plt.close()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

class PerformanceMetrics:
    """Calculate and display trading performance metrics."""

    @staticmethod
    def calculate_metrics(backtest_results, initial_capital=10000):
        """Calculate key performance metrics."""
        if backtest_results.empty or len(backtest_results) == 0:
            return {
                'Total Return (%)': 0.0,
                'Sharpe Ratio': 0.0,
                'Max Drawdown (%)': 0.0,
                'Win Rate (%)': 0.0,
                'Final Value': initial_capital,
                'Total Trades': 0,
            }

        if 'portfolio_value' not in backtest_results.columns:
            return {
                'Total Return (%)': 0.0,
                'Sharpe Ratio': 0.0,
                'Max Drawdown (%)': 0.0,
                'Win Rate (%)': 0.0,
                'Final Value': initial_capital,
                'Total Trades': 0,
            }

        portfolio_values = backtest_results['portfolio_value'].values
        total_return = (portfolio_values[-1] - initial_capital) / initial_capital

        # FIX: Sharpe Ratio Annualization - Corrected for hourly trading intervals (252 days * 24 hours)
        returns = np.diff(portfolio_values) / portfolio_values[:-1]
        sharpe_ratio = np.sqrt(252 * 24) * returns.mean() / (returns.std() + 1e-10)

        # Max Drawdown
        cumulative_max = np.maximum.accumulate(portfolio_values)
        drawdown = (portfolio_values - cumulative_max) / cumulative_max
        max_drawdown = drawdown.min()

        # FIX: Win Rate & Trade Count - Now strictly calculates complete round-trip trades
        trades = []
        entry_price = 0.0
        entry_pos = 0.0
        
        for _, row in backtest_results.iterrows():
            pos = row.get('position', 0)
            price = row.get('price', 0)
            
            if pos != 0 and entry_pos == 0:
                # Open position
                entry_pos = pos
                entry_price = price
            elif pos == 0 and entry_pos != 0:
                # Close position
                pnl = (price - entry_price) / entry_price if entry_pos > 0 else (entry_price - price) / entry_price
                trades.append(pnl)
                entry_pos = 0.0
            elif pos != 0 and entry_pos != 0 and np.sign(pos) != np.sign(entry_pos):
                # Flipped position (e.g., long to short)
                pnl = (price - entry_price) / entry_price if entry_pos > 0 else (entry_price - price) / entry_price
                trades.append(pnl)
                entry_pos = pos
                entry_price = price

        winning_trades = sum(1 for t in trades if t > 0)
        total_trades = len(trades)
        win_rate = winning_trades / total_trades if total_trades > 0 else 0

        return {
            'Total Return (%)': total_return * 100,
            'Sharpe Ratio': sharpe_ratio,
            'Max Drawdown (%)': max_drawdown * 100,
            'Win Rate (%)': win_rate * 100,
            'Final Value': portfolio_values[-1],
            'Total Trades': total_trades,
        }

    @staticmethod
    def plot_results(backtest_results, save_path=None):
        """Plot equity curve and price with trading signals."""
        if backtest_results.empty or len(backtest_results) == 0:
            print("No backtest results to plot.")
            return

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

        # Equity Curve
        if 'portfolio_value' in backtest_results.columns:
            ax1.plot(backtest_results['timestamp'], backtest_results['portfolio_value'],
                     label='Portfolio Value', linewidth=2, color='blue')
            ax1.set_ylabel('Portfolio Value ($)')
            ax1.set_title('Equity Curve')
            ax1.legend()
            ax1.grid(True, alpha=0.3)

        # Price with Trading Signals
        if 'price' in backtest_results.columns:
            ax2.plot(backtest_results['timestamp'], backtest_results['price'],
                     label='Asset Price', linewidth=1, color='black')

            # Mark buy/sell signals based on position changes
            position_diff = backtest_results['position'].diff()
            buy_signals = backtest_results[position_diff > 0]
            sell_signals = backtest_results[position_diff < 0]

            if not buy_signals.empty:
                ax2.scatter(buy_signals['timestamp'], buy_signals['price'],
                           color='green', marker='^', label='Buy Signal', s=50)
            if not sell_signals.empty:
                ax2.scatter(sell_signals['timestamp'], sell_signals['price'],
                           color='red', marker='v', label='Sell Signal', s=50)

            ax2.set_xlabel('Timestamp')
            ax2.set_ylabel('Price ($)')
            ax2.set_title('Trading Signals')
            ax2.legend()
            ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=100)
        plt.show()

    @staticmethod
    def print_report(metrics):
        """Print formatted performance report."""
        print("\n" + "="*50)
        print("BACKTEST PERFORMANCE REPORT")
        print("="*50)
        for key, value in metrics.items():
            if '%' in key:
                print(f"{key}: {value:.2f}%")
            else:
                print(f"{key}: {value:.2f}")
        print("="*50 + "\n")

In [ ]:
#!/usr/bin/env python3
"""
Trading Bot - End-to-end pipeline: download data -> preprocess -> train model -> backtest
"""
import os
import sys
from dotenv import load_dotenv
import pandas as pd
from pathlib import Path

from download_data import download_data, TICKERS, INTERVAL, DATA_PATH
from preprocess import process_all_data, OUTPUT_PATH
from model import PricePredictor
from environment import TradingEnvironment
from backtest import Backtester
from performance import PerformanceMetrics

def main():
    print("="*60)
    print("TRADING BOT - FULL PIPELINE")
    print("="*60)

    # Setup directories safely using Pathlib
    base_dir = Path("trading_bot")
    models_dir = base_dir / "models"
    results_dir = base_dir / "results"
    
    models_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)

    # Phase 1: Download Data
    print("\n[Phase 1] Downloading market data...")
    # download_data(TICKERS, INTERVAL, DATA_PATH)

    # Phase 1: Preprocess Data
    print("\n[Phase 1] Preprocessing data and engineering features...")
    # process_all_data()

    # Phase 2: Train Model
    print("\n[Phase 2] Splitting Data & Training ML model...")
    model = PricePredictor()

    # FIX: Explicitly load the correct master dataset
    load_dotenv()
    MAIN_ASSET = os.getenv("MAIN_ASSET", "EURUSD")
    first_ticker_file = Path(OUTPUT_PATH) / f"{MAIN_ASSET}_master.csv"
    
    if not first_ticker_file.exists():
        print(f"[ERROR] {first_ticker_file} not found. Did you run preprocess.py?")
        sys.exit(1)
        
    # Strictly separate out-of-sample data
    df = pd.read_csv(first_ticker_file, index_col=0, parse_dates=True).iloc[-10_000:]
    split_idx = int(len(df) * 0.8)
    
    train_df = df.iloc[:split_idx]
    
    # FIX: Prevent overlap leakage. By backing up exactly `input_chunk_length`, 
    # the backtester's first trade will land exactly on the first unseen row (split_idx).
    test_df = df.iloc[split_idx - model.input_chunk_length:] 

    train_file = Path(OUTPUT_PATH) / "train_split.csv"
    test_file = Path(OUTPUT_PATH) / "test_split.csv"
    train_df.to_csv(train_file)
    test_df.to_csv(test_file)
    print(train_df.target.value_counts())
    train_score, val_score = model.train(str(train_file))
    print(f"Model trained on 80% split of {first_ticker_file.name}")
    print(f"Internal Validation Score: {val_score:.4f}")

    # Save model safely
    model_path = models_dir / "model.pkl"
    model.save(str(model_path))

    # Plot learning curves safely
    history_path = results_dir / "training_history.png"
    model.plot_training_history(save_path=str(history_path))
    model.plot_confusion_matrix(save_path=str(results_dir / "confusion_matrix.png"))
    # Phase 3: Backtesting
    print("\n[Phase 3] Running out-of-sample backtest...")
    environment = TradingEnvironment(initial_capital=10000)
    backtester = Backtester(model, environment, threshold=0.55, risk_percentage=0.2)

    backtest_results = backtester.run(str(test_file))

    # Calculate metrics
    metrics = PerformanceMetrics.calculate_metrics(
        backtest_results,
        initial_capital=10000
    )
    PerformanceMetrics.print_report(metrics)

    # Plot results safely
    print("Generating performance visualization...")
    viz_path = results_dir / "backtest_results.png"
    PerformanceMetrics.plot_results(backtest_results, save_path=str(viz_path))

    print("\n" + "="*60)
    print("PIPELINE COMPLETE!")
    print("="*60)

if __name__ == "__main__":
    main()